In [1]:
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transform
from torchvision.transforms import Compose
from torch.utils.data import random_split
from PIL import Image

In [2]:
transforms = Compose([
    transform.Resize(256),
    transform.CenterCrop(224),
    # Horizontal flip, default probability is 0.5
    transform.RandomHorizontalFlip(0.5),
    # Random rotation, 10 degrees in either direction
    transform.RandomRotation(10),
    # Color jitter, adjust brightness, contrast, and saturation by a factor of 0.2
    transform.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transform.ToTensor(),
    transform.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

data_set = torchvision.datasets.ImageFolder(root="/Users/damaso/Documents/Proyectos/IA/datasets/premierleagueteams", 
                                             transform=transforms)


In [3]:
classes = data_set.classes
print("Classes Size", len(classes))
print("Classes", classes) 

Classes Size 6
Classes ['arsenal', 'aston-villa', 'chelsea', 'liverpool', 'manchester-city', 'manchester-united']


In [27]:
#data_size = len(data_set)
#train_size = int(0.8 * data_size)
#test_size = data_size - train_size

#train_data, test_data = random_split(data_set, [train_size, test_size])

In [4]:
train_set = torch.utils.data.DataLoader(data_set, batch_size=32, shuffle=True)
#test_set = torch.utils.data.DataLoader(test_data, batch_size=32, shuffle=False, num_workers=4)

In [5]:
len(data_set)

89

In [6]:
class ImageNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, stride=1, padding=1)
        self.batch1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 32, 3, stride=1, padding=1)
        self.batch2 = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(32, 64, 3, stride=1, padding=1)
        self.batch3 = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(64, 64, 3, stride=1, padding=1)
        self.batch4 = nn.BatchNorm2d(64)
        self.conv5 = nn.Conv2d(64, 128, 3, stride=1, padding=1)
        self.batch5 = nn.BatchNorm2d(128)
        self.conv6 = nn.Conv2d(128, 128, 3, stride=1, padding=1)
        self.batch6 = nn.BatchNorm2d(128)

        self.pool = nn.MaxPool2d(2, 2)

        self.dropout = nn.Dropout(0.3)

        self.fc1 = nn.Linear(128 * 28 * 28, 256)
        self.fc2 = nn.Linear(256, 120)
        self.fc3 = nn.Linear(120, 84)
        self.fc4 = nn.Linear(84, 10)
    
    def forward(self, x):
        x = F.relu(self.batch1(self.conv1(x)))
        x = F.relu(self.batch2(self.conv2(x)))
        x = self.pool(x)

        x = F.relu(self.batch3(self.conv3(x)))
        x = F.relu(self.batch4(self.conv4(x)))
        x = self.pool(x)

        x = F.relu(self.batch5(self.conv5(x)))
        x = F.relu(self.batch6(self.conv6(x)))
        x = self.pool(x)
        
        x = self.dropout(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        output = self.fc4(x)
        return output

In [7]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
device

device(type='mps')

In [8]:
net = ImageNet()
net.to(device)

ImageNet(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (batch1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (batch2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (batch3): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv4): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (batch4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (batch5): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv6): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (batch6): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True

In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

In [10]:
for epoch in range(50):

    running_loss = 0.0
    for i, data in enumerate(train_set, 0):
        inputs, labels = data
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        if i % 2 == 1:
            print(f"[{epoch + 1}, {i + 1}] loss: {running_loss / 2:.3f}")
            print(f"{((i + 1) / len(train_set)) * 100:.0f}% from 100%")
            running_loss = 0.0
print("Finished Training")

[1, 2] loss: 2.302
67% from 100%
[2, 2] loss: 2.251
67% from 100%
[3, 2] loss: 2.152
67% from 100%
[4, 2] loss: 2.049
67% from 100%
[5, 2] loss: 2.007
67% from 100%
[6, 2] loss: 1.844
67% from 100%
[7, 2] loss: 1.820
67% from 100%
[8, 2] loss: 1.668
67% from 100%
[9, 2] loss: 1.612
67% from 100%
[10, 2] loss: 1.502
67% from 100%
[11, 2] loss: 1.408
67% from 100%
[12, 2] loss: 1.305
67% from 100%
[13, 2] loss: 1.161
67% from 100%
[14, 2] loss: 1.061
67% from 100%
[15, 2] loss: 0.883
67% from 100%
[16, 2] loss: 0.701
67% from 100%
[17, 2] loss: 0.666
67% from 100%
[18, 2] loss: 0.573
67% from 100%
[19, 2] loss: 0.532
67% from 100%
[20, 2] loss: 0.398
67% from 100%
[21, 2] loss: 0.342
67% from 100%
[22, 2] loss: 0.312
67% from 100%
[23, 2] loss: 0.278
67% from 100%
[24, 2] loss: 0.233
67% from 100%
[25, 2] loss: 0.214
67% from 100%
[26, 2] loss: 0.183
67% from 100%
[27, 2] loss: 0.130
67% from 100%
[28, 2] loss: 0.126
67% from 100%
[29, 2] loss: 0.094
67% from 100%
[30, 2] loss: 0.104
67%

In [11]:
PATH = './models/premier_league_team_logo_clasification_net.pth'

In [12]:

torch.save({"model_state_dict": net.state_dict(), "classes": data_set.classes}, PATH)

In [13]:
net = ImageNet()
#net.load_state_dict(torch.load(PATH, weights_only=True))
#net.eval()
model = torch.load(PATH)
net.load_state_dict(model["model_state_dict"])
net.eval()

ImageNet(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (batch1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (batch2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (batch3): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv4): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (batch4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (batch5): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv6): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (batch6): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True

In [14]:
model["classes"]

['arsenal',
 'aston-villa',
 'chelsea',
 'liverpool',
 'manchester-city',
 'manchester-united']

In [15]:
transforms_val = Compose([
        transform.Resize((256, 256)),
        transform.CenterCrop(224),
        transform.ToTensor(),
        transform.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

In [20]:
img = Image.open("/Users/damaso/Documents/Proyectos/IA/data_prepare/data_test/premier_league_teams/mu.png").convert("RGB")
img = transforms_val(img)
img

tensor([[[-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179],
         [-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179],
         [-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179],
         ...,
         [-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179],
         [-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179],
         [-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179]],

        [[-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357],
         [-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357],
         [-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357],
         ...,
         [-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357],
         [-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357],
         [-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357]],

        [[-1.8044, -1.8044, -1.8044,  ..., -1.8044, -1.8044, -1.8044],
         [-1.8044, -1.8044, -1.8044,  ..., -1

In [21]:
with torch.no_grad():
    y = net(img.unsqueeze(0))

print("Output tensor:", y)
pred = torch.argmax(y)
print("Predicted class index:", pred.item())
print("Predicted class:", data_set.classes[pred])

Output tensor: tensor([[ 0.5893,  2.6227,  0.5249,  2.4252, -0.2737, 10.9273, -7.0932, -4.4894,
         -7.9076, -9.0032]])
Predicted class index: 5
Predicted class: manchester-united
